In [1]:
!pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.2 MB/s eta 0:00:00


In [2]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [14]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

BASE_MODEL = "llama-3.1-8b-instant"

In [21]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Be precise, logical, and code-focused.
Provide debugging steps and example fixes."""
    },
    "billing": {
        "system_prompt": """You are a Billing Support Specialist.
Be empathetic and policy-driven.
Explain charges clearly and outline refund steps."""
    },
    "general": {
        "system_prompt": """You are a helpful Customer Support Assistant.
Handle general inquiries professionally."""
    },
    "tool": {
        "system_prompt": "You are a tool routing expert."
    }
}

In [26]:
def route_prompt(user_input: str) -> str:
    lower_input = user_input.lower()
    if "price" in lower_input and "bitcoin" in lower_input:
        return "tool"

    # Otherwise use LLM classification
    routing_prompt = f"""
Classify the following text into one of these categories:
[technical, billing, general]

Return ONLY the category name.

Text: {user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": "You are a strict classification engine."},
            {"role": "user", "content": routing_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip().lower()

In [27]:
def get_bitcoin_price():
    # Mock function (simulating API call)
    return "$65,432.10 (mock live price)"

In [24]:
def process_request(user_input: str) -> str:
    category = route_prompt(user_input)

    if category not in MODEL_CONFIG:
        category = "general"

    # TOOL HANDLING (Bypass LLM)
    if category == "tool":
        if "bitcoin" in user_input.lower():
            price = get_bitcoin_price()
            return f"[Routed to: TOOL]\n\nCurrent Bitcoin price is {price}"
        else:
            return "[Routed to: TOOL]\n\nRequested live data."

    # Normal LLM flow
    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return f"[Routed to: {category.upper()}]\n\n{response.choices[0].message.content}"

In [18]:
query = "My python script is throwing an IndexError on line 5."
print(process_request(query))

[Routed to: TECHNICAL]

To help you debug the issue, I'll need more information. Please provide the following:

1. The complete error message, including the line number and the error itself.
2. The code snippet where the error is occurring (line 5).
3. The relevant code context, such as the variables and lists used before line 5.

Here's a general approach to troubleshoot the issue:

1. Check the index being used:
   - Is the index valid? (e.g., within the bounds of the list or tuple)
   - Are you using a 0-based index (Python uses 0-based indexing)?

2. Verify the data structure:
   - Are you accessing an element of a list or tuple that is shorter than expected?
   - Have you initialized the data structure correctly?

3. Consider using the `len()` function:
   - You can use `len()` to get the length of a list or tuple, and then use that value to access elements safely.

Here's an example of how to safely access elements in a list:

```python
my_list = [1, 2, 3]

# Safe way to access e

In [19]:
print(process_request("I was charged twice for my subscription."))

[Routed to: BILLING]

I'm so sorry to hear that you were charged twice for your subscription. I'm here to help you understand what happened and provide you with a solution.

To better assist you, can you please provide me with some information? 

1. Your account number or subscription details (if you have them)
2. The dates of the duplicate charges
3. The amounts charged

Once I have this information, I'll review the situation and explain what happened. I'll also provide you with the next steps to resolve the issue.

As a Billing Support Specialist, I want to assure you that we take cases like this seriously and will work with you to get the correct charges adjusted. You won't be charged again for the duplicate amount, and we'll process a refund as soon as possible.

Please rest assured that I'm here to help and will guide you through every step of the process.


In [20]:
print(process_request("Hi, how are you?"))

[Routed to: GENERAL]

I'm doing well, thank you for asking. How can I assist you today? Is there something specific you'd like help with or a question you have?


In [28]:
print(process_request("What is the current price of Bitcoin?"))

[Routed to: TOOL]

Current Bitcoin price is $65,432.10 (mock live price)
